# Partie 3 - Representation des donnees

On repart des donnees deja nettoyees dans le notebook precedent pour les transformer en vecteurs numeriques, seule forme que les modeles de machine learning peuvent utiliser.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from text_cleaning import clean_text

df = pd.read_csv("../data/raw/tweets_suspect.csv")
df = df.dropna(subset=["message", "label"]).drop_duplicates(subset=["message"])
df["clean_text"] = df["message"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0]

print(df.shape)


(59416, 3)


## Pourquoi le TF-IDF

Le sujet propose plusieurs facons de transformer le texte en nombres, des approches simples comme Bag of Words ou TF-IDF, et des approches plus avancees comme Word2Vec ou BERT. On a choisi le **TF-IDF** pour plusieurs raisons :

- Les tweets sont courts, donc pas besoin d'une representation tres complexe pour capturer le sens general.
- TF-IDF ne se contente pas de compter les mots comme le Bag of Words classique : il diminue le poids des mots qui reviennent dans presque tous les tweets, et augmente celui des mots plus specifiques, ce qui aide a mieux distinguer les tweets suspects des autres.
- C'est rapide a calculer et facile a comprendre, contrairement a des embeddings comme BERT qui demandent beaucoup plus de ressources et de temps d'entrainement.
- C'est une bonne base de depart : si les resultats ne sont pas satisfaisants, on pourra toujours essayer une methode plus avancee ensuite.

On a laisse la possibilite d'utiliser des bigrammes en plus des mots seuls (parametre `ngram_max` dans `params.yaml`), pour capturer un peu de contexte (par exemple "not good" est different de "good").

In [2]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["clean_text"])

print("nombre de tweets :", X.shape[0])
print("taille du vocabulaire :", X.shape[1])


nombre de tweets : 59416
taille du vocabulaire : 20000


## Un exemple concret

Pour voir ce que fait le TF-IDF, on regarde les mots avec le score le plus eleve pour quelques tweets.

In [3]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()

for i in range(3):
    row = X[i].toarray().flatten()
    top_idx = row.argsort()[-5:][::-1]
    mots_importants = [feature_names[j] for j in top_idx if row[j] > 0]
    print("tweet :", df["clean_text"].iloc[i])
    print("mots les plus importants selon TF-IDF :", mots_importants)
    print("-" * 60)


tweet : awww bummer shoulda got david carr third day
mots les plus importants selon TF-IDF : ['carr', 'shoulda', 'third', 'david', 'bummer']
------------------------------------------------------------
tweet : upset update facebook texting might cry result school today also blah
mots les plus importants selon TF-IDF : ['today also', 'update facebook', 'texting', 'school today', 'blah']
------------------------------------------------------------
tweet : dived many time ball managed save rest go bound
mots les plus importants selon TF-IDF : ['bound', 'many time', 'managed', 'ball', 'save']
------------------------------------------------------------


## Ce qu'on retient

Le TF-IDF transforme chaque tweet en un vecteur de plusieurs milliers de dimensions (une par mot ou paire de mots du vocabulaire), ou chaque valeur represente l'importance du mot dans ce tweet par rapport a l'ensemble des tweets. C'est cette representation qui sert d'entree aux modeles entraines dans `src/train.py` et qui sera reprise dans la partie suivante pour comparer plusieurs algorithmes de classification.

# Partie 4 - Construction des modeles

On separe les donnees en train/test (meme split que dans le pipeline, test_size=0.2 et random_state=42 pour rester coherent avec `src/preprocess.py`), puis on compare plusieurs modeles de classification.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

print("train :", X_train.shape[0], "| test :", X_test.shape[0])


train : 47532 | test : 11884


## Gestion du desequilibre des classes

On a vu dans l'exploration que les classes sont tres desequilibrees (environ 90% / 10%). On utilise la strategie **class weight** : ca veut dire qu'on dit au modele de donner plus d'importance aux erreurs faites sur la classe minoritaire (les tweets suspects), pour qu'il n'ait pas juste tendance a toujours predire la classe majoritaire. C'est la strategie la plus simple a mettre en place (pas besoin de generer de nouvelles donnees comme avec SMOTE) et elle est directement disponible dans scikit-learn via le parametre `class_weight="balanced"`.

## Comparaison de trois modeles

On compare trois algorithmes differents parmi ceux proposes par le sujet : **Logistic Regression**, **Random Forest** et **Naive Bayes**. Les deux premiers sont les memes que ceux vus en cours (avec en plus Decision Tree qui n'est pas propose dans ce sujet), et Naive Bayes est le modele de base classique pour la classification de texte.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

resultats = []

modeles = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "Naive Bayes": MultinomialNB(),
}

for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    resultats.append({
        "modele": nom,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
    })
    print(nom, "termine")


Logistic Regression termine


Random Forest termine
Naive Bayes termine


Naive Bayes ne prend pas le parametre `class_weight` (ce n'est pas prevu pour ce type de modele), donc pour lui on laisse le desequilibre tel quel, seulement a titre de comparaison.

In [6]:
import pandas as pd

tableau_resultats = pd.DataFrame(resultats).set_index("modele")
tableau_resultats


,accuracy,precision,recall,f1_score
modele,,,,
Logistic Regression,0.975682,0.980091,0.993057,0.986531
Random Forest,0.983087,0.985334,0.995965,0.990621
Naive Bayes,0.915853,0.914222,1.000000,0.955189


## Ce qu'on retient

Les trois modeles donnent des resultats assez proches en accuracy globale, ce qui n'est pas surprenant vu le desequilibre des classes (predire toujours la classe majoritaire donnerait deja une bonne accuracy). Ce qui compte vraiment ici, c'est le recall et le F1-score sur la classe suspecte, puisque c'est elle qu'on cherche a detecter. Le modele avec le meilleur compromis entre les deux sera repris comme modele final dans le pipeline DVC (`params.yaml`).

# Partie 5 - Entrainement et validation

Le split train/test qu'on a utilise jusqu'ici donne un seul resultat, qui peut varier un peu selon la facon dont les donnees ont ete coupees. Pour avoir une idee plus fiable des performances, on utilise une **validation croisee** (cross-validation) : on decoupe les donnees d'entrainement en 5 morceaux (5-fold), et on entraine/teste le modele 5 fois en changeant a chaque fois le morceau utilise comme test. On obtient ainsi une moyenne plus fiable qu'un seul split.

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1"]

resultats_cv = []

for nom, modele in modeles.items():
    scores = cross_validate(modele, X_train, y_train, cv=cv, scoring=scoring)
    resultats_cv.append({
        "modele": nom,
        "accuracy": scores["test_accuracy"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1_score": scores["test_f1"].mean(),
    })
    print(nom, "termine")


Logistic Regression termine


Random Forest termine
Naive Bayes termine


In [8]:
tableau_cv = pd.DataFrame(resultats_cv).set_index("modele")
tableau_cv


,accuracy,precision,recall,f1_score
modele,,,,
Logistic Regression,0.972503,0.977954,0.991696,0.984777
Random Forest,0.980834,0.984351,0.994440,0.989369
Naive Bayes,0.910986,0.909778,0.999906,0.952715


## Ce qu'on retient

Les resultats de la validation croisee sont tres proches de ceux obtenus avec le simple split train/test plus haut, ce qui est plutot rassurant : ca veut dire que les performances ne dependent pas trop de la facon dont les donnees ont ete separees, et que le modele generalise bien.